In [4]:
!pip install azure-ai-projects azure-core azure-storage-blob

In [6]:
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("New_Open_AI")

endpoint = "https://azure-ai-email-123.services.ai.azure.com/api/projects/Default"

client = AIProjectClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

openai_client = OpenAI(
    api_key=api_key,
    base_url=f"{endpoint}/openai/v1",
)

paper_text = input("Paste Paper Text: ")

response = openai_client.responses.create(
    input=[{"role": "user", "content": paper_text}],
    extra_body={
        "agent_reference": {
            "name": "paper-summarizer",
            "version": "1",
            "type": "agent_reference"
        }
    },
)
print(response.output_text)


Paste Paper Text: Title: Efficient Student Performance Prediction Using Machine Learning  Abstract: Predicting student performance is important in education. This paper uses machine learning models to predict pass/fail outcomes.  Method: Dataset of 1000 students with features like attendance, scores, and study hours. Models used: Logistic Regression, Decision Tree, Random Forest.  Results: Random Forest achieved highest accuracy (89%).  Conclusion: Machine learning improves prediction accuracy and helps identify at-risk students early.
**Objective**  
To predict student pass/fail performance using machine learning and identify at-risk students early.

**Method**  
The study used a dataset of 1,000 students with features such as attendance, scores, and study hours. It compared three models: Logistic Regression, Decision Tree, and Random Forest.

**Key Findings**  
Random Forest performed best, achieving **89% accuracy**, outperforming the other models.

**Conclusion**  
Machine learning

In [7]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url=f"{endpoint}/openai/v1",
)

# 1. Define the input paper data
paper_data = """
Title: Efficient Student Performance Prediction Using Machine Learning

Abstract:
Predicting student performance is important in education. This paper uses machine learning models to predict pass/fail outcomes.

Method:
Dataset of 1000 students with features like attendance, scores, and study hours.
Models used: Logistic Regression, Decision Tree, Random Forest.

Results:
Random Forest achieved highest accuracy (89%).

Conclusion:
Machine learning improves prediction accuracy and helps identify at-risk students early.
"""

# 2. Use the Responses API for summarization
try:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Summarize the following research paper into JSON with keys: Objective, Method, Key_Findings, Conclusion. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",
                        "text": f"PAPER SOURCE:\n{paper_data}"
                    }
                ]
            }
        ]
    )

    print("--- SUMMARY ---")
    print(response.output_text)

except Exception as e:
    print(f"Summarization Error: {e}")

--- SUMMARY ---
{"Objective":"To predict student pass/fail performance using machine learning and identify at-risk students early.","Method":"A dataset of 1000 students was used with features such as attendance, scores, and study hours. The study compared Logistic Regression, Decision Tree, and Random Forest models for classification.","Key_Findings":"Random Forest achieved the highest accuracy at 89%, outperforming the other tested models.","Conclusion":"Machine learning can improve student performance prediction accuracy and support early identification of at-risk students."}


In [8]:
import json
import re

try:
    # Get full text output safely
    raw_text = response.output_text

    # Extract JSON portion (in case extra text exists)
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        raise ValueError("No valid JSON found in response")

    json_string = match.group(0)

    # Parse JSON
    extracted_data = json.loads(json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")

--- SUCCESS: DATA PARSED ---
{
  "Objective": "To predict student pass/fail performance using machine learning and identify at-risk students early.",
  "Method": "A dataset of 1000 students was used with features such as attendance, scores, and study hours. The study compared Logistic Regression, Decision Tree, and Random Forest models for classification.",
  "Key_Findings": "Random Forest achieved the highest accuracy at 89%, outperforming the other tested models.",
  "Conclusion": "Machine learning can improve student performance prediction accuracy and support early identification of at-risk students."
}


In [10]:
from azure.storage.blob import BlobServiceClient
from google.colab import userdata
import datetime

blob_conn_str = userdata.get("bcs")

blob_service_client = BlobServiceClient.from_connection_string(blob_conn_str)

file_name = f"paper_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

blob_client = blob_service_client.get_blob_client(
    container="paper-output",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

print("Saved to storage:", file_name)


Saved to storage: paper_20260421_064257.txt
